# Time Series Analysis with ARIMA

Forecasting time-dependent data using classical statistical models.

## Topics Covered
1. Time Series Components (Trend, Seasonality, Residual)
2. Stationarity Testing (ADF Test)
3. Differencing
4. ACF & PACF Plots
5. ARIMA Model (AutoRegressive Integrated Moving Average)
6. Model Evaluation & Forecasting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Generate / Load Time Series Data

In [ ]:
# Generate synthetic time series with trend + seasonality + noise
np.random.seed(42)
n = 365 * 3  # 3 years of daily data
dates = pd.date_range('2022-01-01', periods=n, freq='D')

trend = np.linspace(50, 150, n)
seasonality = 20 * np.sin(2 * np.pi * np.arange(n) / 365)
noise = np.random.normal(0, 5, n)

ts = pd.Series(trend + seasonality + noise, index=dates, name='value')

plt.figure(figsize=(14, 5))
ts.plot(title='Synthetic Time Series (Trend + Seasonality + Noise)', color='steelblue')
plt.ylabel('Value')
plt.tight_layout()
plt.show()
print(f"Shape: {ts.shape}, Range: {ts.index[0]} to {ts.index[-1]}")

## 2. Time Series Decomposition

In [ ]:
decomposition = seasonal_decompose(ts, model='additive', period=365)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
decomposition.observed.plot(ax=axes[0], title='Observed', color='steelblue')
decomposition.trend.plot(ax=axes[1], title='Trend', color='coral')
decomposition.seasonal.plot(ax=axes[2], title='Seasonality', color='green')
decomposition.resid.plot(ax=axes[3], title='Residual', color='gray')
plt.tight_layout()
plt.show()

## 3. Stationarity Test (Augmented Dickey-Fuller)

In [ ]:
def adf_test(series, title=''):
    """Perform ADF test and print results."""
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"=== ADF Test: {title} ===")
    print(f"  Test Statistic: {result[0]:.4f}")
    print(f"  p-value:        {result[1]:.6f}")
    print(f"  Lags Used:      {result[2]}")
    for key, val in result[4].items():
        print(f"  Critical {key}: {val:.4f}")
    if result[1] <= 0.05:
        print("  => Series is STATIONARY ✓")
    else:
        print("  => Series is NON-STATIONARY ✗ (needs differencing)")
    print()

adf_test(ts, 'Original Series')

## 4. Differencing to Achieve Stationarity

In [ ]:
# First difference
ts_diff1 = ts.diff().dropna()

# Second difference
ts_diff2 = ts_diff1.diff().dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
ts.plot(ax=axes[0], title='Original (d=0)', color='steelblue')
ts_diff1.plot(ax=axes[1], title='First Difference (d=1)', color='coral')
ts_diff2.plot(ax=axes[2], title='Second Difference (d=2)', color='green')
plt.tight_layout()
plt.show()

adf_test(ts_diff1, 'First Difference')

## 5. ACF & PACF Plots

Used to determine the order of AR (p) and MA (q) components.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(ts_diff1, ax=axes[0], lags=40, title='ACF (for MA order q)')
plot_pacf(ts_diff1, ax=axes[1], lags=40, title='PACF (for AR order p)',
          method='ywm')

plt.tight_layout()
plt.show()

print("Reading the plots:")
print("  • PACF cuts off after lag p → AR(p)")
print("  • ACF cuts off after lag q → MA(q)")

## 6. ARIMA Model

ARIMA(p, d, q):
- **p**: AR order (from PACF)
- **d**: Differencing order (from ADF test)
- **q**: MA order (from ACF)

In [ ]:
# Train-test split (last 90 days as test)
train = ts[:-90]
test = ts[-90:]

print(f"Train: {train.index[0]} to {train.index[-1]} ({len(train)} days)")
print(f"Test:  {test.index[0]} to {test.index[-1]} ({len(test)} days)")

In [ ]:
# Fit ARIMA model
model = ARIMA(train, order=(2, 1, 2))
fitted = model.fit()

print(fitted.summary())

In [ ]:
# Forecast
forecast = fitted.forecast(steps=90)

# Evaluate
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test - forecast) / test)) * 100

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
# Visualization
plt.figure(figsize=(14, 6))
train[-180:].plot(label='Training Data', color='steelblue')
test.plot(label='Actual', color='green')
forecast.plot(label='ARIMA Forecast', color='coral', linestyle='--')

# Confidence interval
forecast_ci = fitted.get_forecast(steps=90).conf_int(alpha=0.05)
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='coral', alpha=0.15, label='95% CI')

plt.title(f'ARIMA(2,1,2) Forecast — MAE={mae:.2f}, RMSE={rmse:.2f}')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Grid Search for Best ARIMA Order

In [ ]:
best_aic = np.inf
best_order = None
results_list = []

for p in range(0, 4):
    for d in range(0, 3):
        for q in range(0, 4):
            try:
                model = ARIMA(train, order=(p, d, q))
                fitted = model.fit()
                results_list.append({'p': p, 'd': d, 'q': q, 'AIC': fitted.aic})
                if fitted.aic < best_aic:
                    best_aic = fitted.aic
                    best_order = (p, d, q)
            except:
                continue

print(f"Best ARIMA order: {best_order}")
print(f"Best AIC: {best_aic:.2f}")

# Show top 5
results_df = pd.DataFrame(results_list).sort_values('AIC').head(10)
print("\nTop 10 models by AIC:")
print(results_df.to_string(index=False))

## 8. Residual Diagnostics

In [ ]:
best_model = ARIMA(train, order=best_order).fit()
residuals = best_model.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residual plot
residuals.plot(ax=axes[0, 0], title='Residuals Over Time', color='steelblue')
axes[0, 0].axhline(0, color='red', linestyle='--')

# Histogram
residuals.hist(ax=axes[0, 1], bins=30, edgecolor='black')
axes[0, 1].set_title(f'Residual Distribution (mean={residuals.mean():.2f})')

# ACF of residuals
plot_acf(residuals, ax=axes[1, 0], lags=30, title='ACF of Residuals')

# QQ plot
from scipy import stats
stats.probplot(residuals, plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot')

plt.suptitle(f'Residual Diagnostics — ARIMA{best_order}', fontsize=14)
plt.tight_layout()
plt.show()

## Key Takeaways

| Concept | Details |
|---------|---------|
| **Stationarity** | Required for ARIMA — use ADF test |
| **p (AR)** | Determine from PACF cutoff |
| **d (Differencing)** | Number of times to difference for stationarity |
| **q (MA)** | Determine from ACF cutoff |
| **AIC/BIC** | Lower is better — use for model selection |
| **Residuals** | Should be white noise (no autocorrelation) |
| **Extensions** | SARIMA (seasonal), SARIMAX (exogenous), Prophet |